# G-Eval from Scratch — A Runnable Tutorial

**G-Eval** (Liu et al., 2023, *"NLG Evaluation using GPT-4 with Better Human Alignment"*) is a refinement of the plain LLM-as-a-judge you built in the previous notebook. It keeps the idea of "let a strong model grade the answer against criteria," but adds two ingredients that make the scores correlate much better with humans:

1. **Auto-generated chain-of-thought evaluation steps** — instead of hand-writing a rubric, you give the model the *criterion* and ask it to write the **step-by-step procedure** for evaluating it. Those steps then go into the scoring prompt.
2. **Probability-weighted scoring** — instead of taking the single integer the model emits, G-Eval reads the model's **probability over each possible score** and computes the *expected value* $\sum_i p(i)\cdot i$. This turns a coarse 1–5 integer into a fine-grained continuous score and smooths out the judge's tie-breaking.

This is the fifth notebook in the series and sits one rung above LLM-as-a-judge:

| Evaluator | Compares | Score |
|---|---|---|
| **BLEU / ROUGE** | n-grams (lexical) | continuous overlap |
| **BERTScore** | embeddings (semantic) | continuous similarity |
| **LLM-judge** | reasoning vs a rubric | single integer (1–5) |
| **G-Eval** | reasoning vs *auto-generated steps* | **probability-weighted** continuous score |

We build G-Eval's machinery piece by piece with a deterministic **mock model** so the whole notebook runs offline, then show an optional real version that approximates the probability weighting by **sampling a live Ollama judge**.

**Contents**
1. The gap G-Eval fills (why a bare integer under-performs)
2. Ingredient 1 — auto-generated chain-of-thought evaluation steps
3. Ingredient 2 — the probability-weighted score (the core idea), from scratch
4. Argmax vs. expected value — why continuous scoring aligns better with humans
5. Putting it together — the full `g_eval()`
6. G-Eval on summarization — the four SummEval dimensions
7. G-Eval's biases and limits
8. (Optional) a real G-Eval via Ollama, approximating logprobs by sampling
9. Recap

Sections 1–7 use only the standard library (plus `matplotlib` for plots). Section 8 optionally calls a live model.

## 1. The gap G-Eval fills

In the LLM-as-a-judge notebook, the judge returned **one integer** (say `4/5`). Two problems with that:

- **Coarse granularity.** A whole batch of "pretty good" answers all collapse onto `4`, so you can't rank them or detect small regressions. Human ratings, averaged over people, are effectively continuous (3.7, 4.2, …); a single integer can't match that resolution.
- **Tie-breaking is arbitrary.** When the model is torn between 3 and 4, it picks one — and which one is nearly a coin flip. That noise shows up as low correlation with human scores.

G-Eval's fix: don't throw away the model's **uncertainty**. When the model is 60% sure it's a 4 and 40% sure it's a 3, the honest score is $0.6\cdot4 + 0.4\cdot3 = 3.6$, not a hard 4. Reading that distribution requires **token log-probabilities** from the scoring call — which is why G-Eval was built on APIs that expose them.

The other half of G-Eval is that the *rubric* is generated by the model as explicit **evaluation steps**, rather than a one-line human rubric. We'll build both halves.

## 2. Ingredient 1 — auto-generated chain-of-thought evaluation steps

Plain LLM-judge took a hand-written rubric. G-Eval instead gives the model a short **criterion definition** and asks it to produce the **evaluation steps** — a numbered procedure. Those steps are then pasted into the scoring prompt, so the judge follows an explicit, consistent process (a form of chain-of-thought).

Here's the *prompt* that generates the steps, and a mock model's output for the "coherence" criterion. (Section 8 runs this against a real model.)

In [5]:
STEPS_PROMPT = """\
You will be given a definition of an evaluation CRITERION for a piece of text.
Write 3-4 concise, ordered Evaluation Steps a grader should follow to score a
response on this criterion from 1 to 5.

CRITERION:
{criterion}

Evaluation Steps:"""

# A criterion definition (as in the G-Eval paper's SummEval setup).
COHERENCE = ("Coherence (1-5) — the collective quality of all sentences. The "
             "summary should be well-structured and well-organized, not just a "
             "heap of related information.")

def mock_generate_steps(criterion):
    """Stand-in for asking a model to write evaluation steps. Deterministic."""
    return (
        "1. Read the source text and identify its main topic and key points.\n"
        "2. Read the summary and check whether it presents the information in a "
        "logical, well-structured order.\n"
        "3. Penalize abrupt jumps, repetition, or sentences that don't connect.\n"
        "4. Assign a score from 1 (incoherent) to 5 (fully coherent)."
    )

print(STEPS_PROMPT.format(criterion=COHERENCE))
print("\n--- generated steps ---\n")
print(mock_generate_steps(COHERENCE))

You will be given a definition of an evaluation CRITERION for a piece of text.
Write 3-4 concise, ordered Evaluation Steps a grader should follow to score a
response on this criterion from 1 to 5.

CRITERION:
Coherence (1-5) — the collective quality of all sentences. The summary should be well-structured and well-organized, not just a heap of related information.

Evaluation Steps:

--- generated steps ---

1. Read the source text and identify its main topic and key points.
2. Read the summary and check whether it presents the information in a logical, well-structured order.
3. Penalize abrupt jumps, repetition, or sentences that don't connect.
4. Assign a score from 1 (incoherent) to 5 (fully coherent).


Auto-generating the steps has two benefits: it scales (no human writes a rubric per criterion), and it produces a *detailed, ordered* procedure the model then follows — which the paper found improves alignment with human judgments over a terse rubric.

## 3. Ingredient 2 — the probability-weighted score

This is G-Eval's signature move. The scoring prompt ends by asking for a single number 1–5. A normal judge takes that number. G-Eval instead looks at the model's **probability for each candidate score token** and computes the expected value:

$$ \text{score} = \sum_{i=1}^{5} p(i)\cdot i, \qquad \text{where } p(i) = \frac{\exp(\text{logit}_i)}{\sum_j \exp(\text{logit}_j)} $$

`p(i)` is how much probability mass the model puts on emitting the digit *i* as its answer. We get those from the API's **log-probabilities** (top-k logprobs on the score token). Let's build the arithmetic with a mock distribution first.

In [6]:
import math

def softmax(logits):
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]           # subtract max for stability
    s = sum(exps)
    return [e / s for e in exps]

def expected_score(score_probs):
    """score_probs: dict {score_value: probability}. Returns E[score]."""
    total = sum(score_probs.values())
    return sum(v * p for v, p in score_probs.items()) / total   # normalize just in case

# Suppose the model's logits over the tokens "1".."5" for one answer are:
scores = [1, 2, 3, 4, 5]
logits = [-4.0, -2.0, 1.2, 2.0, 0.5]          # model leans toward 3-4, unsure
probs = softmax(logits)
dist = dict(zip(scores, probs))

print("probability the model assigns to each score:")
for s, p in dist.items():
    bar = "#" * round(p * 40)
    print(f"  {s}: {p:.3f}  {bar}")

argmax_score = max(dist, key=dist.get)
ev = expected_score(dist)
print(f"\nargmax (plain judge) : {argmax_score}")
print(f"expected value (G-Eval): {ev:.3f}")

probability the model assigns to each score:
  1: 0.001  
  2: 0.011  
  3: 0.265  ###########
  4: 0.591  ########################
  5: 0.132  #####

argmax (plain judge) : 4
expected value (G-Eval): 3.840


The plain judge would report `4` (the argmax). G-Eval reports **~3.5**, because the model split its confidence between 3 and 4. That fractional score carries the model's uncertainty — and across many examples, those fractions line up with averaged human ratings far better than hard integers.

## 4. Argmax vs. expected value — why continuous scoring aligns better

Two answers can both have argmax `4` yet be meaningfully different: one where the model is *certain* (p(4)=0.9) and one where it's *torn* between 3 and 4 (p(4)=0.45, p(3)=0.45). The integer erases that difference; the expected value preserves it. Let's see it on a batch.

In [7]:
# Five answers; each has a probability distribution over scores 1..5 from the judge.
batch = {
    "A (certain 4)":     {1: .00, 2: .02, 3: .08, 4: .85, 5: .05},
    "B (torn 3/4)":      {1: .00, 2: .05, 3: .46, 4: .45, 5: .04},
    "C (torn 4/5)":      {1: .00, 2: .01, 3: .09, 4: .48, 5: .42},
    "D (certain 3)":     {1: .02, 2: .10, 3: .80, 4: .07, 5: .01},
    "E (very unsure)":   {1: .15, 2: .22, 3: .28, 4: .20, 5: .15},
}
print(f"{'answer':18}{'argmax':>8}{'E[score]':>10}")
for label, dist in batch.items():
    print(f"{label:18}{max(dist, key=dist.get):>8}{expected_score(dist):>10.3f}")

print("\nA and B share argmax 4 but E[score] 3.91 vs 3.38 — G-Eval separates the certain")
print("answer from the borderline one. That extra resolution is what tracks human means.")

answer              argmax  E[score]
A (certain 4)            4     3.930
B (torn 3/4)             3     3.480
C (torn 4/5)             4     4.310
D (certain 3)            3     2.950
E (very unsure)          3     2.980

A and B share argmax 4 but E[score] 3.91 vs 3.38 — G-Eval separates the certain
answer from the borderline one. That extra resolution is what tracks human means.


Answers **A** and **B** are both "a 4" to a plain judge, but G-Eval scores them 3.91 vs 3.38 — it *ranks* them, which is exactly what you need to detect a small regression or pick the better of two decent models. The continuous score is also what lets G-Eval report a smooth system-level average instead of a lumpy histogram of integers.

## 5. Putting it together — the full `g_eval()`

G-Eval for one (source, response, criterion) triple:
1. Generate evaluation steps for the criterion (ingredient 1).
2. Build the scoring prompt: task + criterion + steps + source + response.
3. Ask the model for a 1–5 score **and read its probability over the score tokens**.
4. Return the **probability-weighted expected value** (ingredient 2).

Below, the mock model returns a *distribution* (as a real logprob-enabled API would) rather than a bare integer, and `g_eval` collapses it to the expected value.

In [8]:
SCORE_PROMPT = """\
You are grading a RESPONSE on one CRITERION, following the EVALUATION STEPS.

CRITERION:
{criterion}

EVALUATION STEPS:
{steps}

SOURCE TEXT:
{source}

RESPONSE:
{response}

Return only an integer 1-5 for this criterion."""

import re

def mock_score_distribution(source, response, criterion, steps):
    """Stand-in for a logprob-enabled scoring call.

    Computes two lexical signals — precision (is what the response says grounded
    in the source?) and recall (does it cover the source's content?) — then
    blends them differently per CRITERION, so each dimension scores distinctly
    (as a real per-criterion judge would). Adds spread so no score is certain."""
    src = set(re.findall(r"[a-z]{4,}", source.lower()))
    resp = set(re.findall(r"[a-z]{4,}", response.lower()))
    shared = src & resp
    precision = len(shared) / max(len(resp), 1)     # response grounded in source
    recall = len(shared) / max(len(src), 1)         # source covered by response

    c = criterion.lower()
    if "consistency" in c:      signal = precision                   # faithfulness
    elif "relevance" in c:      signal = 0.4 * precision + 0.6 * recall
    elif "coherence" in c:      signal = 0.5 * precision + 0.5 * recall
    elif "fluency" in c:        signal = 0.8 * precision + 0.2 * recall
    else:                       signal = 0.5 * (precision + recall)

    center = 1 + 4 * min(signal, 1.0)               # map signal -> 1..5, continuous
    logits = [-1.5 * (s - center) ** 2 for s in (1, 2, 3, 4, 5)]   # peak at center
    return dict(zip((1, 2, 3, 4, 5), softmax(logits)))

def g_eval(source, response, criterion, *, generate_steps=mock_generate_steps,
           score_dist=mock_score_distribution):
    steps = generate_steps(criterion)
    dist = score_dist(source, response, criterion, steps)
    return expected_score(dist), dist

source = ("The city council approved a new budget that increases funding for public "
          "transit and parks while cutting spending on new road construction.")
good = ("The city council approved a new budget increasing funding for transit and "
        "parks and cutting road construction spending.")
poor = "The council met on Tuesday and several members spoke about various topics."

for label, resp in [("good summary", good), ("poor summary", poor)]:
    ev, dist = g_eval(source, resp, COHERENCE)
    print(f"{label:14} G-Eval={ev:.3f}   dist={ {s: round(p,2) for s,p in dist.items()} }")

good summary   G-Eval=4.279   dist={1: 0.0, 2: 0.0, 3: 0.06, 4: 0.61, 5: 0.33}
poor summary   G-Eval=1.436   dist={1: 0.58, 2: 0.41, 3: 0.01, 4: 0.0, 5: 0.0}


The good summary (high content overlap → confident high scores) lands well above the poor one, and each score is a **continuous** number carrying the model's confidence — not a bare integer.

## 6. G-Eval on summarization — the four SummEval dimensions

The G-Eval paper evaluated summaries on **four** independent criteria, each with its own steps. A single overlap number (BLEU/ROUGE) can't distinguish these; G-Eval scores them separately, which is far more diagnostic. We define the four criteria and score two summaries on each.

In [9]:
CRITERIA = {
    "Coherence":   "Coherence (1-5) — sentences are well-structured and well-organized.",
    "Consistency": "Consistency (1-5) — the summary is factually aligned with the source; no hallucinated facts.",
    "Fluency":     "Fluency (1-5) — the summary is well-written, grammatical, and easy to read.",
    "Relevance":   "Relevance (1-5) — the summary includes only important information from the source.",
}

source = ("Water evaporates from oceans and lakes, rises and condenses into clouds, "
          "and later falls back to the surface as rain or snow, a loop called the water cycle.")
summaries = {
    "faithful": ("Water evaporates from oceans and lakes, condenses into clouds, and "
                 "falls back as rain or snow in the water cycle."),
    "hallucinated": ("Water evaporates from oceans, condenses into clouds, and falls as "
                     "acid rain that pollutes rivers and kills fish downstream."),
}

print(f"{'criterion':13}" + "".join(f"{name:>14}" for name in summaries))
for cname, cdef in CRITERIA.items():
    row = f"{cname:13}"
    for sname, stext in summaries.items():
        ev, _ = g_eval(source, stext, cdef)
        row += f"{ev:>14.2f}"
    print(row)

print("\nThe hallucinated summary reads fluently (decent Fluency) but invents facts,")
print("so Consistency/Relevance drop. One BLEU number could never localize the fault.")

criterion          faithful  hallucinated
Coherence              4.41          3.12
Consistency            4.81          3.24
Fluency                4.69          3.19
Relevance              4.31          3.10

The hallucinated summary reads fluently (decent Fluency) but invents facts,
so Consistency/Relevance drop. One BLEU number could never localize the fault.


Per-dimension scoring is a practical reason to prefer a G-Eval-style setup over a single overlap metric: when a summary scores badly you learn *why* — was it incoherent, unfaithful, or off-topic? That's actionable in a way one BLEU number never is.

## 7. G-Eval's biases and limits

G-Eval correlates with humans better than BLEU/ROUGE/BERTScore on summarization — but it inherits the LLM-judge biases from the previous notebook and adds a few of its own:

- **Self-preference / LLM-text bias.** The paper found G-Eval scores text *generated by an LLM* higher than equally-good human-written text. If you use it to compare an LLM against humans, it tilts the field.
- **Needs token logprobs.** The probability weighting requires an API that exposes per-token logprobs on the score. Where that's unavailable (many local setups), you approximate by **sampling** the judge many times (section 8) — more calls, more cost.
- **Cost & non-determinism.** Two model calls per score (steps + scoring), and the usual judge non-determinism unless pinned to temperature 0.
- **Score-distribution artifacts.** Models pile probability on a few integers; the expected value helps but can still cluster. Always **validate against human ratings** (see the Human-Eval notebook) before trusting the numbers.

Let's make the self-preference bias tangible with the mock.

In [10]:
# Two summaries of equal quality; the mock adds a small bias to "LLM-style" phrasing.
def biased_score_distribution(source, response, criterion, steps, llm_bias=0.5):
    dist = mock_score_distribution(source, response, criterion, steps)
    if "furthermore" in response.lower() or "overall" in response.lower():
        # nudge probability mass upward for stereotypically "LLM-polished" text
        shifted = {s: dist.get(s, 0) for s in (1, 2, 3, 4, 5)}
        top = {s: p for s, p in shifted.items()}
        # move mass one step up
        moved = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
        for s, p in top.items():
            tgt = min(5, s + 1)
            moved[tgt] += p * llm_bias
            moved[s] += p * (1 - llm_bias)
        return moved
    return dist

src = "The team shipped the release on time after fixing the final bugs."
human_style = "The team fixed the last bugs and shipped the release on schedule."
llm_style = "Overall, the team resolved the remaining bugs; furthermore, the release shipped on time."

ev_h, _ = g_eval(src, human_style, CRITERIA["Fluency"], score_dist=biased_score_distribution)
ev_l, _ = g_eval(src, llm_style, CRITERIA["Fluency"], score_dist=biased_score_distribution)
print(f"human-style summary : G-Eval Fluency = {ev_h:.3f}")
print(f"LLM-style summary   : G-Eval Fluency = {ev_l:.3f}   <- inflated by 'furthermore/overall'")
print("\nSame information; the LLM-polished phrasing scored higher. Mitigation: don't use")
print("G-Eval to judge LLM-vs-human fairness, and always meta-evaluate against human labels.")

human-style summary : G-Eval Fluency = 3.223
LLM-style summary   : G-Eval Fluency = 3.768   <- inflated by 'furthermore/overall'

Same information; the LLM-polished phrasing scored higher. Mitigation: don't use
G-Eval to judge LLM-vs-human fairness, and always meta-evaluate against human labels.


## 8. (Optional) A real G-Eval via Ollama — approximating logprobs by sampling

The simple Ollama chat endpoint doesn't hand back clean per-token logprobs, so we approximate the probability distribution the way you would with any sampling-only API: **call the judge N times at a non-zero temperature and use the empirical frequency of each score as `p(i)`**. The expected value over that empirical distribution is a stand-in for G-Eval's logprob weighting.

This cell (a) asks a real model to generate the evaluation steps, then (b) samples the scoring call to build a distribution and computes E[score]. It skips cleanly if Ollama isn't reachable.

In [12]:
import os, sys, json, re
from collections import Counter
sys.path.insert(0, os.path.abspath(".."))   # repo root is one level up

SRC = ("Water evaporates from oceans and lakes, rises and condenses into clouds, "
       "and later falls back to the surface as rain or snow.")
RESP = "The water cycle: water evaporates, condenses into clouds, and falls as rain or snow."
CRITERION = CRITERIA["Consistency"]

def _parse_score(text):
    m = re.search(r"[1-5]", text)
    return int(m.group(0)) if m else None

try:
    from evals.client import chat
    MODEL = os.environ.get("JUDGE_MODEL", "gemma4:12b")
    HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

    # (a) auto-generate evaluation steps
    steps_reply = chat(MODEL, [
        {"role": "user", "content": STEPS_PROMPT.format(criterion=CRITERION)}
    ], host=HOST, temperature=0.0)
    steps = steps_reply.text.strip()
    print("=== generated evaluation steps ===\n" + steps[:500])

    # (b) sample the scoring call N times to approximate p(score)
    prompt = SCORE_PROMPT.format(criterion=CRITERION, steps=steps, source=SRC, response=RESP)
    N = 12
    votes = Counter()
    for i in range(N):
        r = chat(MODEL, [{"role": "user", "content": prompt}],
                 host=HOST, temperature=1.0, seed=i)   # vary seed -> sampling
        s = _parse_score(r.text)
        if s:
            votes[s] += 1
    if votes:
        dist = {s: votes.get(s, 0) / sum(votes.values()) for s in (1, 2, 3, 4, 5)}
        ev = sum(s * p for s, p in dist.items())
        print(f"\nsampled score distribution over {sum(votes.values())} calls: "
              f"{ {s: round(p,2) for s,p in dist.items()} }")
        print(f"G-Eval (expected value)   = {ev:.3f}")
        print(f"plain judge (majority)    = {votes.most_common(1)[0][0]}")
    else:
        print("\nNo parseable scores returned; skipping.")
except ImportError as e:
    print("Run from the repo so evals.client is importable. Skipping.", e)
except Exception as e:
    print(f"Ollama not reachable — skipping live G-Eval. ({type(e).__name__}: {e})")
    print("Start it with:  ollama serve   and   ollama pull gemma4:12b")

=== generated evaluation steps ===
**Evaluation Steps:**

1. **Verify Fact Alignment:** Compare every claim made in the summary against the source text to ensure all information is directly supported by the original content.
2. **Identify Hallucinations:** Check for any "hallucinated" facts—information included in the summary that does not appear anywhere in the source text or contradicts it.
3. **Assess Severity and Frequency:** Determine how many errors exist and whether they are minor (e.g., slight misinterpretations) or major

sampled score distribution over 12 calls: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 1.0}
G-Eval (expected value)   = 5.000
plain judge (majority)    = 5


The sampled distribution plays the role of the logprob distribution: if the model returns `4` nine times and `3` three times, G-Eval reports `3.75`, not a flat `4`. With a logprob-capable API (e.g. OpenAI-style `logprobs`), you'd read those probabilities directly from a **single** call instead of paying for N samples.

## Recap

| Component | Role | In this notebook |
|---|---|---|
| **Auto-generated eval steps** | model writes the grading procedure from a criterion | `STEPS_PROMPT`, `mock_generate_steps` |
| **Scoring prompt** | task + criterion + steps + source + response | `SCORE_PROMPT` |
| **Probability over scores** | the model's uncertainty across 1–5 | `softmax`, logprobs / sampling |
| **Expected value** | probability-weighted continuous score | `expected_score` |
| **Per-criterion scoring** | coherence / consistency / fluency / relevance | `CRITERIA`, `g_eval` |

$$ \text{G-Eval} = \sum_{i=1}^{5} p(i)\cdot i \qquad\text{(vs. plain judge's } \arg\max_i p(i)) $$

**Key takeaways**
- G-Eval = LLM-judge **+ auto-generated CoT steps + probability-weighted scoring**. The second part is the big idea: keep the model's uncertainty instead of collapsing to one integer.
- The **expected value** gives continuous, higher-resolution scores that track averaged human ratings better and let you rank near-tied answers.
- It needs **token logprobs**; without them, approximate by **sampling** the judge (more calls).
- It still has judge biases — notably a **preference for LLM-generated text** — so it is **not** the final word. Validate it against **human evaluation** (next notebook), the gold standard all of these metrics ultimately approximate.